In [ ]:
import sys
#!pip install keplergl --target ./my_custom_packages
sys.path.append('./my_custom_packages') 
# Polars, fastexcel, hvplot, "vegafusion[embed]>=1.5.0", ""vl-convert-python>=1.6.0"", xgboost, graphviz, geopandas, setuptools<81
# keplergl, 

import polars as pl
import pandas as pd
import polars.selectors as cs
import statsmodels.api as sm
from statsmodels.formula.api import ols
import numpy as np

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt
import plotly.express as px

import geopandas as gpd
import plotly.graph_objects as go
from keplergl import KeplerGl
import shapely

from scipy.stats import chi2_contingency


 

In [ ]:
df_merged = pl.read_parquet("df_merged.parquet")
display(df_merged.head())

In [ ]:
%%time

# Regression lineaire et anova

df_pd = df_merged.to_pandas()
display(sum(df_pd["Max_RAT"] == "2G"))
# Less than 1% of all entries

df_pd = df_pd[df_pd["Max_RAT"] != "2G"]

lm_r = ols("CA ~ nb_jours + C(Group_canal) + C(Max_RAT) + C(Typologie)", data = df_pd).fit()
display(lm_r.summary())

# CA in terms of Max_RAT
anova_max_rat = sm.stats.anova_lm(ols('CA ~ C(Max_RAT)', data = df_pd).fit(), typ = 2)
display(anova_max_rat)


In [ ]:
%%time

# Regression lineaire et anova WITH CA LOG SCALE

df_pd["CA_log"] = np.log(df_pd["CA"] + 1)


lm_r_log = ols("CA_log ~ nb_jours + C(Group_canal) + C(Max_RAT) + C(Typologie)", data = df_pd).fit()
display(lm_r_log.summary())

# CA in terms of Max_RAT
anova_max_rat_log = sm.stats.anova_lm(ols('CA_log ~ C(Max_RAT)', data = df_pd).fit(), typ = 2)
display(anova_max_rat_log)


In [ ]:
# Baseline
display(round(((2.71828)**6.7719 - 1.0), 2))

In [ ]:
# PARETO check

df_ca = df_merged.group_by(["msisdn"]).agg(pl.col("CA").sum().alias("Sum_CA"))

df_pd_ca = df_ca.to_pandas()

vip_threshold = df_pd_ca["Sum_CA"].describe(percentiles=[0.5, 0.8, 0.9, 0.95, 0.99])

display(vip_threshold)


total_revenue = df_pd_ca["Sum_CA"].sum()

rev_top_20 = df_pd_ca[df_pd_ca["Sum_CA"] >= 8600]["Sum_CA"].sum()
print(f"Top 20% spenders: {(rev_top_20 / total_revenue) * 100:.1f}% of revenue")

rev_top_5 = df_pd_ca[df_pd_ca["Sum_CA"] >= 22100]["Sum_CA"].sum()
print(f"Top 5% spenders: {(rev_top_5 / total_revenue) * 100:.1f}% of revenue")


user_top_20 = df_pd_ca[df_pd_ca["Sum_CA"] >= 8600]["msisdn"].tolist()

print(f"Top 20% user: {(df_merged.filter(pl.col("msisdn").is_in(user_top_20)).shape[0] / df_merged.shape[0]) * 100:.1f}% of traffic")

user_top_5 = df_pd_ca[df_pd_ca["Sum_CA"] >= 22100]["msisdn"].tolist()

print(f"Top 5% user: {(df_merged.filter(pl.col("msisdn").is_in(user_top_5)).shape[0] / df_merged.shape[0]) * 100:.1f}% of traffic")

In [ ]:
# XGBOOST classifier prep

df_pd = df_merged.to_pandas()

df_pd_ca["VIP"] = (df_pd_ca["Sum_CA"] >= 8600).astype(int)

df_xgboost = df_pd.merge(df_pd_ca[["msisdn", "VIP"]], on = "msisdn")

y = df_xgboost["VIP"]

x = pd.get_dummies(df_xgboost[["nb_jours", "Group_canal", "Typologie", "Max_RAT"]], drop_first = False)


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 4, stratify = y)


In [ ]:
%%time

xgb_model = xgb.XGBClassifier(
    n_estimators = 150,
    max_depth = 6,
    learning_rate = 0.1,
    n_jobs = -1,
    tree_method = 'hist',
    random_state = 4 
)


xgb_model.fit(x_train, y_train)

# Gain measures the improvement in accuracy a feature brings to the branches it splits

xgb.plot_importance(xgb_model, importance_type = 'gain')
plt.title("Feature xgboost importance score in determining VIP category (top 20% spending user)")
plt.show()

y_predic = xgb_model.predict(x_test)
display(classification_report(y_test, y_predic))

In [ ]:
print(classification_report(y_test, y_predic))

In [ ]:
# Why are most big spenders using 3G networks?


# M2M (Machine-to-Machine)
# High volume transaction devices like  
# Point of Sale (POS)terminals, mobile money merchant phones, and older dual-SIM business phones
# default to 3G. They don't need 4G broadband speeds to send few kilobyte Orange Money transaction 

# Voice and USSD fallback
# Unless a user has a specific VoLTE (Voice over LTE) setup 
# making a phone call or dialing a USSD code forces the phone to temporarily drop from 4G down to 3G. 
# If VIPs are constantly making business calls and doing USSD transfers, 
# their network logs will have their most connected site to be 3G.

In [ ]:

xgb.plot_tree(xgb_model, tree_idx = 0, rankdir = "LR")
plt.show()

tree_graph = xgb.to_graphviz(xgb_model, num_trees = 0, rankdir = "LR")
display(tree_graph)

tree_text = xgb_model.get_booster().get_dump()[0]
display(tree_text)

In [ ]:
# Are vip more concentrated in frequent 3G cell users than in frequent 4G/5G users?

vip_table = df_xgboost.groupby(["Max_RAT", "Typologie"]).agg(
    Total_activity_rows = ("msisdn", "count"),
    Total_VIP_rows = ("VIP", "sum")
)

vip_table["VIP_concentration_%"] = (vip_table["Total_VIP_rows"] / vip_table["Total_activity_rows"]) * 100

# Sort by the highest concentration to see what floats to the top
vip_table = vip_table.sort_values("VIP_concentration_%", ascending=False)

# VIPs by data technology and geography
display(vip_table)

In [ ]:
# What canal are those users choosing?

urban_high_speed_df = df_xgboost[
    (df_xgboost["Max_RAT"].str.contains("4G|5G")) & 
    (df_xgboost["Typologie"].isin(["Urbain", "Peri-Urbain"]))
]

channel_check = urban_high_speed_df.groupby("Group_canal").agg(
    Total_activity_rows=("msisdn", "count"),
    Total_VIP_rows=("VIP", "sum")
)

channel_check["VIP_concentration_%"] = (channel_check["Total_VIP_rows"] / channel_check["Total_activity_rows"]) * 100

channel_check = channel_check.sort_values("VIP_concentration_%", ascending=False)

display(channel_check)

# Digital channels MAXIT + APP OM + WEB, VIP Rows: ~950,000
# Legacy channels USSD + IVR, VIP Rows: ~4,770,000
# Among the wealthiest highest speed urban users, 83% of their VIP activity is still happening on USSD and IVR.

In [ ]:
# Does better data technology trigger high value data package spending? 



product_determinants = pd.crosstab(
    index = df_xgboost["Max_RAT"],
    columns = df_xgboost["Gamme_groupe"],
    values = df_xgboost["CA"],
    aggfunc = ["count", "mean"]
)

display(product_determinants)

# Infrastructure upgrades do change what people buy
# Infrastructure change how much they consume

In [ ]:
# Does site infrastructure explain customer stability and value?


site_strategy_analysis = df_xgboost.groupby(["Typologie site", "Typologie"]).agg(
    Avg_days_connected = ("nb_jours", "mean"),
    Avg_spend_per_row = ("CA", "mean"),
    Total_revenue = ("CA", "sum")
).sort_values(by = "Avg_spend_per_row", ascending = False)

display(site_strategy_analysis)

In [ ]:
# Do geographic regions explain revenue density and channel preference?

regional_segmentation = df_xgboost.groupby("sig_region_name").agg(
    Total_transactions = ("msisdn", "count"),
    Total_CA = ("CA", "sum"),
    Preferred_channel = ("Group_canal", lambda x: x.value_counts().index[0]),
    Most_used_forfait = ("Nom du forfait", lambda x: x.value_counts().index[0])
).sort_values(by = "Total_CA", ascending=False)

display(regional_segmentation)


# Analamanga: Prefers USSD and buys "Akama Plus" Data centric

# (Diana not anymore), Sava, Boeny (Coastal): Rely on IVR and buy "Be 500 New" voice.

# IVR dominates throughout

In [ ]:
# voix et data

display(df_merged.head())


# AKAMA and BE CONNECT are data plans
# BE offers are voice plan with a small data bonus
# 3g usage is automatically set when calling someone.

# Are voice and data region specific? Maybe people on data use mainly social apps for phone calls
# Do voice and data differ in geographic locations? Differ across cities?
# Do voice and data differ in group canal choice?


display(df_merged.select(pl.col("sig_region_name").str.to_lowercase())["sig_region_name"].unique().to_list())

display(df_merged.filter(pl.col("sig_region_name") == "")["msisdn"].count())
display(df_merged.filter(pl.col("sig_region_name") == "")["msisdn"].unique().count())

# ~300k transactions without a region
# ~50k users involved

In [ ]:
mada_gpd = gpd.read_file("MDG_adm/MDG_adm2.shp")
mgpd = sorted(mada_gpd["NAME_2"].unique().tolist())

#display(mgpd)
#display(len(mgpd))


# (?i): case sensitive
# ($): exact match

df_gpd = (
    df_merged
    .filter(pl.col("sig_region_name") != "")
    .with_columns(
        pl.when(pl.col("sig_region_name").str.contains("(?i)^atsinanana$"))
        .then(pl.lit("Atsinanana"))
        .when(pl.col("sig_region_name").str.contains("(?i)vatovavy|fitovinany"))
        .then(pl.lit("Vatovavy Fitovinany"))
        .when(pl.col("sig_region_name").str.contains("(?i)atsinanana"))
        .then(pl.lit("Atsimo-Atsinana"))
        .when(pl.col("sig_region_name").str.contains("(?i)andrefana"))
        .then(pl.lit("Atsimo-Andrefana"))
        .when(pl.col("sig_region_name").str.contains("(?i)mangoro"))
        .then(pl.lit("Alaotra-Mangoro"))
        .when(pl.col("sig_region_name").str.contains("(?i)fitovinany"))
        .then(pl.lit("Vatovavy Fitovinany"))
        .when(pl.col("sig_region_name").str.contains("(?i)mania"))
        .then(pl.lit("Amoron'i mania"))
        .when(pl.col("sig_region_name").str.contains("vambony"))
        .then(pl.lit("Haute matsiatra"))
        .when(pl.col("sig_region_name").str.contains("(?i)matsiatra"))
        .then(pl.lit("Haute matsiatra"))
        .when(pl.col("sig_region_name").str.contains("(?i)vatovavy"))
        .then(pl.lit("Vatovavy Fitovinany"))
        .when(pl.col("sig_region_name").str.contains("(?i)vatovavy"))
        .then(pl.lit("Vatovavy Fitovinany"))

        .otherwise(pl.col("sig_region_name").str.to_titlecase())
        .alias("region_cleaned")
    )
)

dfgpd = df_gpd["region_cleaned"].unique().sort().to_list()
#display(dfgpd)
#display(len(dfgpd))

display("Perfect match") if (mgpd == dfgpd) else display("Mismatch")


df_gpd = (
    df_gpd
    .group_by("region_cleaned")
    .agg(pl.col("Gamme_groupe").mode().first().alias("Gamme_groupe"))
)
df_gpd.head()

In [ ]:
# Are Voice and Data region-specific?



# 0.01 degrees = 1 km. 
simplified_geoms = shapely.simplify(mada_gpd["geometry"].values, tolerance = 0.01, preserve_topology = True)

mada_gpd["geometry"] = simplified_geoms


choropleth = px.choropleth(
    df_gpd.to_pandas(),
    geojson = mada_gpd.__geo_interface__,
    locations = "region_cleaned",
    featureidkey = "properties.NAME_2",
    title = "Mobile plans by region",
    color = "Gamme_groupe"
)

choropleth.update_geos(fitbounds = "locations", visible = False)
choropleth.write_html("madagascar_mobile_plans.html")

In [ ]:
df_kepler = df_merged.to_pandas()
gdf_regions = gpd.read_file("MDG_adm/MDG_adm2.shp")

map_madagascar = KeplerGl(height = 600)


map_madagascar.add_data(data = gdf_regions, name = "Boundaries")
map_madagascar.add_data(data = df_kepler, name = "orange_mai_data")

map_madagascar.save_to_html(file_name = "orange_kepler_map.html")

In [ ]:
gdf_regions = gpd.read_file("MDG_adm/MDG_adm2.shp")
gdf_regions["geometry"] = shapely.simplify(gdf_regions["geometry"].values, tolerance=0.01, preserve_topology=True)

df_kepler = (
    df_merged
    .group_by(["sig_nom_site", "x", "y", "Max_RAT"])
    .agg([
        pl.col("CA").sum().alias("Total_CA"),
        pl.len().alias("n_user")
    ])
    .to_pandas(use_pyarrow_extension_array = True)
)


map_madagascar = KeplerGl()

map_madagascar.add_data(data=gdf_regions, name = "Boundaries")
map_madagascar.add_data(data=df_kepler, name = "Orange_data")

map_madagascar.save_to_html(file_name = "orange_kepler.html")

df_kepler.to_csv("orange_site_data.csv")

In [ ]:
css_patch = """
<style>
    html, body, #app, .kepler-gl, #root, [class^="kepler-gl"] {
        width: 100vw !important;
        height: 100vh !important;
        position: absolute !important;
        top: 0 !important;
        left: 0 !important;
        margin: 0 !important;
        padding: 0 !important;
    }
</style>
"""

with open("orange_kepler.html", "a", encoding = "utf-8") as f:
    f.write(css_patch)


In [ ]:
df_major_cities = (
    df_merged
    .group_by("region_cleaned")
    .agg([
        pl.col("x").first(),
        pl.col("y").first(),
        pl.col("CA").sum()
    ])
    .sort("CA", descending=True)
    .head(15)
    .to_pandas()
)

display(df_major_cities)
df_major_cities.to_csv("major_cities.csv", index = False)


In [ ]:
# Do voice and data differ in group canal choice?

df_flows = df_merged.filter(pl.col("Gamme_groupe").is_in(["Be", "Akama/Be Connect", "Be Sms"])).group_by(["Gamme_groupe", "Group_canal"]).agg(pl.len().alias("count"))

all_nodes = pl.concat([df_flows["Gamme_groupe"], df_flows["Group_canal"]]).unique().to_list()

source_indices = [all_nodes.index(item) for item in df_flows["Gamme_groupe"]]
destination_indices = [all_nodes.index(item) for item in df_flows["Group_canal"]]
values = df_flows["count"].to_list()

sankey = go.Figure(
    data = [
        go.Sankey(
            arrangement = "snap", # Organizes the nodes
            node = dict(label = all_nodes, pad = 20),
            link = dict(
                source = source_indices,
                target = destination_indices,
                value = values
            )
        )
    ]
)

sankey.update_layout(
    title_text = " Do voice and data differ in group canal choice?",
)



sankey.show()


In [ ]:
%%time

# Do certain regions have a statistically significant bias toward data or voice?

df_merged_filered = df_merged.filter(pl.col("Gamme_groupe").is_in(["Be", "Akama/Be Connect", "Be Sms"]))

matrix = pd.crosstab(df_merged_filered["Gamme_groupe"], df_merged_filered["region_cleaned"])
chi2, p, dofree, exp = chi2_contingency(matrix)


# Cramer's V number
n = matrix.sum().sum()
min_dim = min(matrix.shape) - 1
cramer_v = np.sqrt(chi2 / (n * min_dim))

display(p)
display(f"{cramer_v:.4f}")
# The result is moderate. The fields are moderately associated.

percentages = pd.crosstab(df_merged_filered["region_cleaned"], df_merged_filered["Gamme_groupe"], normalize = "index") * 100
display(percentages.sort_values(by = "Akama/Be Connect", ascending = False).round(1))


